# 1. Setup and Configuration
นำเข้า Library และกำหนดค่าพื้นฐานสำหรับการรัน EDA โดยคุณสามารถเลือกเปิด/ปิด แต่ละส่วนได้ผ่านตัวแปร `RUN_...` และตรวจสอบ Path ของข้อมูลที่ `PROJECT_ROOT`

In [ ]:
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})
sns.set_theme(style="whitegrid")

# --- Configuration ---
RUN_DATA_QUALITY = True
RUN_SPARSITY = True
RUN_COVERAGE_DRIFT = True
RUN_PROMO_DEPTH = True
RUN_DISCOUNT_ELASTICITY = True
RUN_PRODUCT_LIFECYCLE = True
RUN_STOCKOUT_SPELLS = True
RUN_CUSTOMER_TENURE = True

PROJECT_ROOT = Path(r"C:\Users\CPE KMUTT\Documents\GitHub\superai_engineer_ss6\Level 2\Hackathon 5_Demand Forecasting Coffee Chain Hackathon")
OUTPUT_DIR = PROJECT_ROOT / "eda_additional_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_base() -> Path:
    candidates = [
        PROJECT_ROOT / "super-ai-engineer-season-6-coffee-chain-hackathon",
        Path.cwd() / "super-ai-engineer-season-6-coffee-chain-hackathon",
        Path("/kaggle/input/competitions/super-ai-engineer-season-6-coffee-chain-hackathon"),
        Path("/kaggle/input/super-ai-engineer-season-6-coffee-chain-hackathon"),
    ]
    for candidate in candidates:
        if candidate.exists(): return candidate
    raise FileNotFoundError("Dataset folder not found.")

def save_table(df: pd.DataFrame, name: str) -> None:
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

def save_fig(name: str) -> None:
    path = OUTPUT_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path)
    plt.show() 
    plt.close()
    print(f"Saved: {path}")

# 2. Data Loading and Processing
โหลดข้อมูลจาก CSV ทั้งหมด และเตรียม DataFrame พื้นฐานสำหรับการวิเคราะห์ (Line-level และ Daily-level)

In [ ]:
def load_data(base: Path) -> dict:
    train = base / "train"
    test = base / "test"
    data = {
        "txn": pd.read_csv(train / "TRANSACTION.csv"),
        "order": pd.read_csv(train / "ORDER.csv", parse_dates=["date"]),
        "prod": pd.read_csv(train / "PRODUCT.csv"),
        "store": pd.read_csv(train / "STORE.csv", parse_dates=["opened_date"]),
        "promo": pd.read_csv(train / "PROMOTION.csv", parse_dates=["start_date", "end_date"]),
        "cust": pd.read_csv(train / "CUSTOMER.csv", parse_dates=["registration_date"]),
        "date_dim": pd.read_csv(train / "DATE_DIM.csv", parse_dates=["date"]),
        "event": pd.read_csv(train / "LOCAL_EVENT.csv", parse_dates=["date"]),
        "inventory": pd.read_csv(train / "INVENTORY.csv", parse_dates=["date"]),
        "test_date": pd.read_csv(test / "DATE_DIM.csv", parse_dates=["date"]),
        "test_promo": pd.read_csv(test / "PROMOTION.csv", parse_dates=["start_date", "end_date"]),
        "test_event": pd.read_csv(test / "LOCAL_EVENT.csv", parse_dates=["date"]),
        "test_prod": pd.read_csv(test / "PRODUCT.csv"),
        "test_store": pd.read_csv(test / "STORE.csv", parse_dates=["opened_date"]),
    }
    sample_path = base / "sample_submission_with_id.csv"
    data["sample_submission"] = pd.read_csv(sample_path) if sample_path.exists() else None
    return data

def build_line(data: dict) -> pd.DataFrame:
    txn = data["txn"]; order = data["order"]; prod = data["prod"]
    order_cols = [c for c in ["order_id", "store_id", "date", "customer_id", "hour", "is_member", "payment_method"] if c in order.columns]
    prod_cols = [c for c in ["product_id", "category", "base_price", "is_seasonal", "is_limited_edition", "serve_type"] if c in prod.columns]
    line = txn.merge(order[order_cols], on="order_id", how="left")
    line = line.merge(prod[prod_cols], on="product_id", how="left")
    return line

def build_daily(line: pd.DataFrame) -> pd.DataFrame:
    agg_map = {"units_sold": ("units_sold", "sum"), "n_orders": ("order_id", "nunique"), "n_customers": ("customer_id", "nunique")}
    if "revenue" in line.columns: agg_map["revenue"] = ("revenue", "sum")
    else: agg_map["revenue"] = ("units_sold", "count")
    daily = line.groupby(["store_id", "category", "date"], observed=True).agg(**agg_map).reset_index()
    return daily

def build_daily_full(daily: pd.DataFrame, data: dict) -> pd.DataFrame:
    store_ids = sorted(data["store"]["store_id"].dropna().unique())
    categories = sorted(data["prod"]["category"].dropna().unique())
    all_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
    idx = pd.MultiIndex.from_product([store_ids, categories, all_dates], names=["store_id", "category", "date"])
    out = daily.set_index(["store_id", "category", "date"]).reindex(idx, fill_value=0).reset_index()
    return out

# 3. Data Quality Checks
ตรวจสอบความซ้ำซ้อนของข้อมูล (Duplicates) และความเชื่อมโยงของ Table ต่างๆ (Missing Foreign Keys) รวมถึงค่าที่ผิดปกติ เช่น ราคาติดลบ หรือ ชั่วโมงเกิน 23 น.

In [ ]:
def data_quality_checks(data: dict, line: pd.DataFrame) -> None:
    results = []
    for tab in ["order", "txn", "prod", "store"]:
        df = data[tab]
        id_col = tab+"_id" if tab != "txn" else "transaction_id"
        if id_col in df.columns:
            results.append({"check": f"duplicates_{tab}", "count": int(df.duplicated(subset=[id_col]).sum())})
    
    if "order_id" in data["order"].columns and "order_id" in data["txn"].columns:
        missing_orders = data["order"].loc[~data["order"]["order_id"].isin(data["txn"]["order_id"]), "order_id"].nunique()
        results.append({"check": "orders_missing_transactions", "count": int(missing_orders)})

    results_df = pd.DataFrame(results)
    print(results_df.sort_values("count", ascending=False))
    save_table(results_df, "data_quality_checks")

# 4. Zero Inflation (Sparsity) Analysis
วิเคราะห์ดูว่ามีกี่วันที่ยอดขายเป็น 0 ในแต่ละสาขาและหมวดหมู่สินค้า (สำคัญมากสำหรับ Demand Forecasting)

In [ ]:
def zero_inflation(daily_full: pd.DataFrame) -> None:
    zero_rate = (daily_full.groupby(["store_id", "category"], observed=True)["units_sold"]
                 .apply(lambda s: (s == 0).mean()).reset_index(name="zero_rate"))
    save_table(zero_rate.sort_values("zero_rate", ascending=False), "zero_rate_by_store_category")
    plt.figure(figsize=(10, 4))
    sns.histplot(zero_rate["zero_rate"], bins=30, color="#457b9d")
    plt.title("Zero-rate distribution across store-category")
    save_fig("zero_rate_distribution")

# 5. Coverage and Drift Analysis
เช็คว่าใน Test set มี Event หรือ Promotion ประเภทใหม่ๆ ที่ไม่เคยเจอใน Train set หรือไม่ และเช็คความหนาแน่นของข้อมูลเทียบกัน

In [ ]:
def expand_promo(promo_df: pd.DataFrame, product_df: pd.DataFrame) -> pd.DataFrame:
    if promo_df.empty: return pd.DataFrame(columns=["store_id", "category", "date", "promo_id", "discount_pct", "promo_type"])
    p = promo_df.merge(product_df[["product_id", "category"]], on="product_id", how="left").copy()
    p["date"] = p.apply(lambda r: pd.date_range(r["start_date"], r["end_date"], freq="D"), axis=1)
    p = p.explode("date")
    return p

def coverage_drift(data: dict, daily: pd.DataFrame) -> None:
    train_promos = set(data["promo"]["promo_type"].dropna().unique())
    test_promos = set(data["test_promo"]["promo_type"].dropna().unique())
    new_promos = sorted(test_promos - train_promos)
    save_table(pd.DataFrame({"new_promo_types_in_test": new_promos}), "new_promo_types_in_test")
    print(f"New promo types: {new_promos}")

# 6. Promotion and Elasticity Analysis
วิเคราะห์ว่า Promotion ซ้อนกันหนักแค่ไหน (Overlap) และวัดความอ่อนไหวของยอดขายต่อส่วนลด (Discount Elasticity) ในแต่ละหมวดหมู่สินค้า

In [ ]:
def promo_depth_and_overlap(data: dict) -> None:
    promo_daily = expand_promo(data["promo"], data["prod"])
    if promo_daily.empty: return
    store_day = promo_daily.groupby(["store_id", "date"], observed=True).agg(promo_campaign_count=("promo_id", "nunique")).reset_index()
    plt.figure(figsize=(10, 4))
    sns.histplot(store_day["promo_campaign_count"], bins=20, color="#2a9d8f")
    plt.title("Promo overlap per store-day")
    save_fig("promo_overlap_hist")

def discount_elasticity(data: dict, daily: pd.DataFrame) -> None:
    promo_daily = expand_promo(data["promo"], data["prod"])
    if promo_daily.empty: return
    promo_cat = promo_daily.groupby(["store_id", "category", "date"], observed=True).agg(max_discount=("discount_pct", "max")).reset_index()
    merged = daily.merge(promo_cat, on=["store_id", "category", "date"], how="left").fillna(0)
    merged["discount_bin"] = pd.cut(merged["max_discount"], bins=[-0.1, 0, 10, 25, 50, 100], labels=["0", "0-10", "10-25", "25-50", ">50"])
    summary = merged.groupby(["category", "discount_bin"], observed=True)["units_sold"].mean().unstack()
    plt.figure(figsize=(12, 6))
    sns.heatmap(summary, annot=True, fmt=".1f", cmap="YlGnBu")
    plt.title("Avg units sold by Discount Bin")
    save_fig("discount_elasticity_heatmap")

# 7. Lifecycle and Stockout Analysis
ดูวงจรชีวิตสินค้า (SKU Lifecycle) ว่าชิ้นไหนมาใหม่หรือเลิกขายไปแล้ว และวิเคราะห์การขาดสต็อก (Stockout Spells) ว่ายาวนานแค่ไหน

In [ ]:
def product_lifecycle(line: pd.DataFrame, data: dict) -> None:
    sku = line.groupby(["product_id", "category"], observed=True).agg(first_sale=("date", "min"), last_sale=("date", "max"), units=("units_sold", "sum")).reset_index()
    sku["days_since_last_sale"] = (line["date"].max() - sku["last_sale"]).dt.days
    save_table(sku.sort_values("units", ascending=False), "sku_lifecycle_summary")
    plt.figure(figsize=(10, 4))
    sns.histplot(sku["days_since_last_sale"], bins=30, color="#e76f51")
    plt.title("Days since last sale (SKU)")
    save_fig("sku_days_since_last_sale")

def stockout_spells(data: dict) -> None:
    inv = data["inventory"].copy()
    if "is_stockout" not in inv.columns: return
    inv["is_stockout_int"] = inv["is_stockout"].astype(int)
    inv = inv.sort_values(["store_id", "product_id", "date"])
    inv["spell_id"] = inv.groupby(["store_id", "product_id"], observed=True)["is_stockout_int"].diff().ne(0).cumsum()
    spells = inv[inv["is_stockout_int"].eq(1)].groupby(["store_id", "product_id", "spell_id"], observed=True).size().reset_index(name="stockout_days")
    save_table(spells.sort_values("stockout_days", ascending=False), "stockout_spells")

# 8. Customer Tenure Analysis
วิเคราะห์พฤติกรรมลูกค้าตามระยะเวลาที่เป็นสมาชิก (Tenure) เพื่อดูว่าลูกค้าเก่ากับลูกค้าใหม่มียอดสั่งซื้อต่างกันอย่างไร

In [ ]:
def customer_tenure(line: pd.DataFrame, data: dict) -> None:
    order = data["order"].copy(); cust = data["cust"].copy()
    if "customer_id" not in order.columns or "customer_id" not in cust.columns: return
    merged = order.merge(cust[["customer_id", "registration_date"]], on="customer_id", how="left")
    merged["tenure_days"] = (merged["date"] - merged["registration_date"]).dt.days
    merged["tenure_bin"] = pd.cut(merged["tenure_days"], bins=[-1, 90, 365, 730, 5000], labels=["0-90", "91-365", "366-730", ">730"])
    summary = merged.groupby("tenure_bin", observed=True).size().reset_index(name="orders")
    plt.figure(figsize=(10, 4))
    sns.barplot(data=summary, x="tenure_bin", y="orders", color="#1d3557")
    plt.title("Orders by Customer Tenure")
    save_fig("orders_by_tenure")

# 9. Main Execution
รันฟังก์ชันทั้งหมดตามลำดับ ผลลัพธ์จะถูกแสดงใน Notebook และบันทึกลงในโฟลเดอร์ `eda_additional_outputs`

In [ ]:
def main() -> None:
    base = find_base()
    data = load_data(base)
    line = build_line(data)
    daily = build_daily(line)
    daily_full = build_daily_full(daily, data)

    if RUN_DATA_QUALITY: data_quality_checks(data, line)
    if RUN_SPARSITY: zero_inflation(daily_full)
    if RUN_COVERAGE_DRIFT: coverage_drift(data, daily)
    if RUN_PROMO_DEPTH: promo_depth_and_overlap(data)
    if RUN_DISCOUNT_ELASTICITY: discount_elasticity(data, daily)
    if RUN_PRODUCT_LIFECYCLE: product_lifecycle(line, data)
    if RUN_STOCKOUT_SPELLS: stockout_spells(data)
    if RUN_CUSTOMER_TENURE: customer_tenure(line, data)

if __name__ == "__main__":
    main()